# Chẩn đoán theo cây giả thuyết: vì sao lượt thuê tháng 10/2012 giảm 9%?

Bộ `bike-sharing-uci`, `day.csv`: 731 ngày (01/01/2011 tới 31/12/2012) của Capital Bikeshare, Washington DC.
`cnt` là tổng lượt thuê, tách `casual` (khách vãng lai) và `registered` (thành viên); `weathersit` 1 = quang,
2 = mây/sương, 3 = mưa hoặc tuyết nhẹ, 4 = mưa to.

**Bối cảnh.** Đầu tháng 11/2012, giám đốc vận hành nhắn: *"Lượt thuê tháng 10 giảm 9% so với tháng 9. Có phải
khách bắt đầu bỏ dịch vụ không? Tôi cần biết trước họp ban điều hành thứ Năm."*

Đây là câu hỏi **diagnostic**. Thay vì mở dữ liệu ra tìm ngẫu nhiên, bạn viết cây giả thuyết (phần 4 lý thuyết)
**trước**, mỗi nhánh một con số kiểm và một ngưỡng, rồi kiểm lần lượt:

| Nhánh | Giả thuyết | Con số kiểm | Nếu ... thì nhánh này giải thích được |
| --- | --- | --- | --- |
| Lịch | Hai tháng khác số ngày | Lượt thuê **mỗi ngày** | Mức giảm mỗi ngày nhỏ hơn hẳn 9% |
| Mùa vụ | Tháng 10 năm nào cũng thấp hơn tháng 9 | Cùng chuyển tháng năm 2011 | Năm 2011 cũng giảm cỡ tương tự |
| Thời tiết | Tháng 10/2012 xấu trời hơn | Số ngày quang / xấu; so **trong cùng loại thời tiết** | Trong cùng loại thời tiết mức giảm nhỏ |
| Phân khúc | Một nhóm khách bỏ đi | Đóng góp của `casual` và `registered` vào mức giảm | Một nhóm chiếm phần lớn bất thường |

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/01-analyst-mindset/exercises/intermediate/02-hypothesis-driven-diagnosis -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc day.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
day = pd.read_csv(BIKE / "day.csv")
print(day.shape, day.dteday.min(), "→", day.dteday.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Bảng tháng: nhánh lịch, mùa vụ và đếm thời tiết

`month_summary(day) -> pd.DataFrame`: mỗi tháng lịch của `dteday` một dòng, index ngày đầu tháng
(`DatetimeIndex` tên `month`, tăng dần). Bảy cột theo thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `rides` | tổng `cnt` |
| `casual`, `registered` | tổng từng cột |
| `days` | số ngày (số dòng) của tháng trong dữ liệu |
| `per_day` | `rides / days` |
| `clear_days` | số ngày `weathersit == 1` |
| `bad_days` | số ngày `weathersit >= 3` |

In [ ]:
def month_summary(day: pd.DataFrame) -> pd.DataFrame:
    """Index month; cột rides, casual, registered, days, per_day, clear_days, bad_days."""
    # TODO: tháng = pd.to_datetime(dteday).dt.to_period("M").dt.to_timestamp()
    raise NotImplementedError

In [ ]:
# explore: nhánh lịch và nhánh mùa vụ
summary = month_summary(day)
print(summary.loc["2011-08-01":"2012-12-01"].round(1).to_string())
for m0, m1 in [("2012-09-01", "2012-10-01"), ("2011-09-01", "2011-10-01")]:
    a, b = summary.loc[m0], summary.loc[m1]
    print(f"{m0[:7]} → {m1[:7]}: tổng {b.rides / a.rides - 1:+.1%}, mỗi ngày {b.per_day / a.per_day - 1:+.1%}, "
          f"ngày quang {int(a.clear_days)} → {int(b.clear_days)}, ngày xấu {int(a.bad_days)} → {int(b.bad_days)}")

## 2. Nhánh phân khúc

`segment_contribution(day, m0, m1) -> pd.DataFrame`: index `["casual", "registered"]`, năm cột theo thứ tự:
`before` (tổng tháng `m0`), `after` (tổng tháng `m1`), `change` (`after - before`), `pct` (`after / before - 1`),
`share_of_change` (`change` chia tổng `change` của hai nhóm). `m0`, `m1` là chuỗi ngày đầu tháng như
`"2012-09-01"`.

In [ ]:
def segment_contribution(day: pd.DataFrame, m0: str, m1: str) -> pd.DataFrame:
    """Index casual, registered; cột before, after, change, pct, share_of_change."""
    # TODO: dùng lại month_summary
    raise NotImplementedError

## 3. Nhánh thời tiết: so trong cùng loại thời tiết

`same_weather_change(day, m0, m1) -> pd.Series`: với mỗi `weathersit` **có mặt ở cả hai tháng**, lượt thuê trung
bình mỗi ngày của tháng `m1` chia cho tháng `m0`, trừ 1. Index là `weathersit` (tăng dần), tên Series `change`.

So trong cùng loại thời tiết trả lời: "nếu thời tiết giống nhau, tháng 10 có thấp hơn tháng 9 không?". Nếu mức
giảm trong cùng loại thời tiết nhỏ, phần lớn mức giảm tổng đến từ việc tháng 10 có nhiều ngày xấu trời hơn
(hiệu ứng **mix** thời tiết).

In [ ]:
def same_weather_change(day: pd.DataFrame, m0: str, m1: str) -> pd.Series:
    """TB cnt mỗi ngày theo weathersit, tháng m1 so với m0, trừ 1; chỉ weathersit có ở cả hai tháng."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kiểm nhánh phân khúc và thời tiết cho năm 2012 và năm 2011
for m0, m1 in [("2012-09-01", "2012-10-01"), ("2011-09-01", "2011-10-01")]:
    print(f"== {m0[:7]} → {m1[:7]}")
    print(segment_contribution(day, m0, m1).round(3))
    print(same_weather_change(day, m0, m1).round(3))

In [ ]:
# explore: nếu tháng 10/2012 có cơ cấu thời tiết như tháng 9 thì lượt mỗi ngày là bao nhiêu?
month = pd.to_datetime(day["dteday"]).dt.to_period("M").dt.to_timestamp()
sep, oct_ = day[month == "2012-09-01"], day[month == "2012-10-01"]
sep_mix = sep.weathersit.value_counts(normalize=True)
oct_mean = oct_.groupby("weathersit").cnt.mean()
oct_at_sep_mix = (oct_mean.reindex(sep_mix.index) * sep_mix).sum()
print("cơ cấu thời tiết tháng 9:", sep_mix.round(3).to_dict(), "| tháng 10:", oct_.weathersit.value_counts(normalize=True).round(3).to_dict())
print(f"tháng 10 với thời tiết của tháng 9: {oct_at_sep_mix:,.0f} lượt/ngày, so với tháng 9 {sep.cnt.mean():,.0f} "
      f"→ {oct_at_sep_mix / sep.cnt.mean() - 1:+.1%}")

# bão Sandy: bỏ 29 và 30/10/2012
dates = pd.to_datetime(oct_.dteday)
no_sandy = oct_[~dates.isin(pd.to_datetime(["2012-10-29", "2012-10-30"]))]
print(f"tháng 10 bỏ 29-30/10: {no_sandy.cnt.mean():,.0f} lượt/ngày → {no_sandy.cnt.mean() / sep.cnt.mean() - 1:+.1%} so với tháng 9")
print(oct_[dates >= "2012-10-27"][["dteday", "weathersit", "casual", "registered", "cnt"]].to_string(index=False))

In [ ]:
# explore: một chart cho giám đốc - lượt mỗi ngày tháng 9 và tháng 10, tô ngày xấu trời
both = day[month.isin([pd.Timestamp("2012-09-01"), pd.Timestamp("2012-10-01")])].assign(date=lambda d: pd.to_datetime(d.dteday))
colors = both.weathersit.map({1: "#4c78a8", 2: "#9aa5b1", 3: "#e45756"})
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(both.date, both.cnt, color=colors, width=0.8)
ax.annotate("Bão Sandy 29-30/10", xy=(pd.Timestamp("2012-10-29"), 300), xytext=(-150, 120),
            textcoords="offset points", arrowprops={"arrowstyle": "->"}, fontsize=9)
ax.set_ylabel("lượt thuê / ngày")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Tháng 10/2012: nhiều ngày mây hơn và hai ngày bão; ngày quang gần như không đổi", loc="left", fontsize=11)
plt.show()

## 4. Kết luận cho giám đốc, rồi làm trắc nghiệm

Điền bảng cây giả thuyết ở đầu notebook bằng con số bạn tìm được: nhánh nào giải thích được, nhánh nào bị loại.
Rồi viết ba câu cho giám đốc theo thứ tự **kết luận → bằng chứng → giới hạn**: khách có đang bỏ dịch vụ không,
con số nào chứng minh, và điều gì phân tích này chưa trả lời được (ví dụ dữ liệu ngày không có thông tin số
thành viên đang hoạt động).

Xong thì mở app, làm phần **trắc nghiệm** của bài.